# 🧠 Claude Agent SDK — "Give Claude a Computer"

> *"The same engine that powers Claude Code, programmable in your own apps."*

**By Anthropic** · Python & TypeScript · Powers **Claude Code** · Managed Agents in public beta (2026)

---

## Table of Contents
1. [What is the Claude Agent SDK?](#1)
2. [Core Philosophy: Give Claude a Computer](#2)
3. [Agent SDK vs Client SDK](#3)
4. [The Tool-Use System](#4)
5. [Multi-Agent Pattern: Planner / Generator / Evaluator](#5)
6. [Conceptual Code](#6)
7. [How the Claude Agent SDK is Advanced vs Other Frameworks](#7)
8. [Strengths, Weaknesses & When to Use](#8)


<a id="1"></a>
## 1. What is the Claude Agent SDK?

The **Claude Agent SDK** is the **same infrastructure that powers Claude Code**, exposed for your own applications. It gives you Claude's **tools, agent loop, and context management** — programmable in **Python and TypeScript** — and works well far beyond coding.

The key distinction from a raw API: **the SDK runs the agent loop for you.** Claude autonomously decides which tool to call, executes it, observes the result, and iterates — you don't hand-write the loop.

> In 2026 Anthropic shipped two related things: the **Claude Agent SDK** (a harness for self-hosted agents you run) and **Managed Agents** (hosted agent infrastructure, public beta April 2026).


<a id="2"></a>
## 2. Core Philosophy: Give Claude a Computer

The SDK's guiding idea is **"give Claude a computer"** — real capabilities, not just chat:

```
┌──────────────────────────────────────────────┐
│              CLAUDE AGENT SDK                 │
│                                               │
│   🖥️  Bash execution   (run real commands)    │
│   📂  File system       (read / write files)  │
│   🔌  MCP integrations  (connect any tool/API)│
│   🔁  Agent loop        (Claude drives itself)│
│   🧠  Context mgmt       (compaction, memory) │
└──────────────────────────────────────────────┘
```

Because it operates at the **project level**, a Claude agent can read an entire codebase, plan across many files, execute changes, run tests, and **iterate on failures** — autonomously.


<a id="3"></a>
## 3. Agent SDK vs Client SDK

A crucial difference that defines the SDK:

| | **Client SDK** (`anthropic`) | **Agent SDK** |
|---|---|---|
| **Tool loop** | *You* implement it | **Claude handles it autonomously** |
| **You write** | The full think→act→observe loop | Just the tools + goal |
| **Context mgmt** | Manual | **Built-in** (compaction, memory) |
| **Best for** | Fine control over each API call | Autonomous, multi-step agents |

> With the **Client SDK** you orchestrate every tool call yourself. With the **Agent SDK**, you describe tools and a goal, and Claude runs the loop end-to-end.


<a id="4"></a>
## 4. The Tool-Use System

Anthropic's tool-use system is unusually **mature** in 2026:

| Capability | What it does |
|---|---|
| **Parallel tool calls** | Multiple `tool_use` blocks in one response → run tools concurrently |
| **Dynamic tool discovery** | `tool_search` finds tools on demand, avoiding **50,000+ tokens** of upfront tool definitions |
| **Strict schema enforcement** | Tool inputs are validated against their schema |
| **Fine-grained streaming** | Per-tool streaming of inputs/outputs |
| **Permissions** | Explicit human approval **required** before file edits or commands |

> The **dynamic tool discovery** matters at scale: instead of stuffing every tool definition into the prompt, Claude searches for the right tool when it needs it — saving huge amounts of context.


<a id="5"></a>
## 5. Multi-Agent Pattern: Planner / Generator / Evaluator

In an April 2026 engineering post, Anthropic detailed a **production-validated** pattern for long-running agents — divide the work across three roles:

```
        ┌──────────────┐
        │   PLANNER    │  defines structure & goals
        └──────┬───────┘
               │ hands off a plan
               ▼
        ┌──────────────┐
        │  GENERATOR   │  executes the work
        └──────┬───────┘
               │ produces output
               ▼
        ┌──────────────┐
        │  EVALUATOR   │  independent quality check
        └──────┬───────┘
               │ pass? → done   |   fail? → back to Generator
               ▼
            Result
```

The **independent evaluator** is the trick: a separate agent judging quality catches errors the generator is blind to — a reliability pattern that scales to long-horizon tasks.


<a id="6"></a>
## 6. Conceptual Code

```python
# Python — the Agent SDK runs the loop for you
from claude_agent_sdk import query, ClaudeAgentOptions

options = ClaudeAgentOptions(
    model="claude-opus-4-8",
    # "Give Claude a computer": allow file + shell tools, ask before risky actions
    allowed_tools=["Read", "Write", "Bash"],
    permission_mode="acceptEdits",   # or require approval per action
)

# Describe the GOAL — Claude plans, calls tools, observes, and iterates itself
async for message in query(
    prompt="Find every TODO in this repo and turn them into a checklist in TODO.md",
    options=options,
):
    print(message)

# No hand-written think→act→observe loop:
# the SDK owns tool execution, context compaction, and iteration.
```

> Compare to the raw Client SDK, where *you* would write the `while` loop, dispatch each `tool_use`, append results, and manage context yourself.


<a id="7"></a>
## 7. How the Claude Agent SDK is Advanced vs Other Frameworks

| Dimension | Claude Agent SDK | LangGraph | CrewAI | OpenAI Agents SDK |
|---|---|---|---|---|
| **Agent loop** | **Built-in, model-native** | You build the graph | Process engine | Built-in |
| **"Computer" access** | **Native Bash + files + MCP** | Via custom tools | Via tools | Via tools/sandbox |
| **Context management** | **Automatic compaction/memory** | Manual / checkpointer | Basic memory | Sessions |
| **Tool scaling** | **Dynamic tool_search** (saves tokens) | Manual | Manual | Manual |
| **Safety** | **Permission gates** before actions | DIY | DIY | Guardrails |
| **Lock-in** | Anthropic models | Open | Open | OpenAI models |

### What makes it "advanced"
1. **It's the engine behind a real product (Claude Code).** Battle-tested on huge tasks — e.g. a 50,000-line Python→Go migration in ~20 hours that was estimated at 2–3 months.
2. **Model-native agent loop + context management.** Tool execution, iteration, and *context compaction* are handled by infrastructure tuned to the model — not bolted on.
3. **Dynamic tool discovery** avoids the "50k tokens of tool defs" problem, so agents stay efficient even with huge tool catalogs.
4. **Safety by default.** Explicit permission before edits/commands, operating in *your* environment rather than an opaque backend — a deliberate trust design.


<a id="8"></a>
## 8. Strengths, Weaknesses & When to Use

### ✅ Strengths
- **Most capable autonomous loop** out of the box (it powers Claude Code)
- Native **Bash + file system + MCP** — real "computer" access
- Automatic **context management** for long tasks
- Strong **safety/permission** model; Python **and** TypeScript

### ⚠️ Weaknesses
- **Tied to Anthropic models** (not model-agnostic)
- Less of a *multi-agent orchestration* DSL than CrewAI/LangGraph (you compose patterns yourself)
- Newest as a general-purpose SDK — ecosystem still forming

### 🎯 Use the Claude Agent SDK when…
- You want **maximum autonomous capability** with minimal loop-writing
- The task needs **real tool use** — files, shell, codebases, MCP services
- You're building on **Claude** and value the safety/permission model

> **Rule of thumb:** *If you'd otherwise be hand-writing a tool loop around the Claude API, the Agent SDK already does it better — and it's the closest thing to "Claude Code as a library."*

---

> **Sources**
> - [Agent SDK overview — Claude Code Docs](https://code.claude.com/docs/en/agent-sdk/overview)
> - [Building agents with the Claude Agent SDK — Anthropic Engineering](https://www.anthropic.com/engineering/building-agents-with-the-claude-agent-sdk)
> - [Claude Code — Anthropic](https://www.anthropic.com/product/claude-code)
> - [Claude Agent SDK Capabilities & Ecosystem Guide — AI Agents Hub](https://www.aiagentshub.net/blog/claude-agent-sdk-guide)
> - [Anthropic Agent SDK: What It Ships vs What You Build — Augment Code](https://www.augmentcode.com/guides/anthropic-agent-sdk-what-ships-vs-what-you-build)
> - [Claude Agent SDK & Managed Agents Architecture — Zylos Research](https://zylos.ai/research/2026-04-20-claude-agent-sdk-managed-agents-architecture/)
